# Deteccion de cuellos de botella

### Lo que queremos es encontrar los famosos cuellos de botella en el flujo de facturación
### ¿Cuál es el flujo? (en 'Happy path')
### Alta médica → Liquidación → Primera factura → Alta administrativa

### Cuando no funciona de una manera facil (en 'un happy path')
### Alta médica → ... → Primera factura → Revisión/ajuste/abono → Segunda factura → ... → Alta administrativa

### Las tablas que vamos a usar:
#### 1.- Hospac: Tiene los datos generales (podria decirse que es la tabla central)
#### 2.- Hosfol: Tiene los folios/expedientes por lo que es el puente administrativo (fin e inicio de un folio)
#### 3.- Hosffa: Contiene las fechas de las facturas (primera y ultima factura) osea ciclos de facturación
#### 4.- Hostransacciones: Cuenta con las transacciones (Cargos, abonos, ajustes, cancelaciones, movimientos negativos y facturados.)
#### 5.- Hoshab: Aun si no están relacionadas los tipos de camas y habitaciones y los retrasos en el flujo de facturación (puede ayudar a encontrar problemas (patrones) en distintos perfiles del hospital
#### 6.- Hosreq: Tiene las solicitudes, aperturas, montos, departamento solicitante y relacion de episodios
#### 7.- Hosder: Contiene a detalle las requesiciones osea diferencias pedido vs surtido, devoluciones
#### 8.- Hostha: nombre legible del tipo de aviation (ayuda igual que Hoshab)

In [4]:
# Importamos los dataframes
import pandas as pd
from src.hospital_app.data_loader import dfs, nuevas_llaves

## Estrategia analitica-exploratoria
### Usaremos 5 layers (podemos verlas como 'etapas') basadas en el proceso de facturación
#### 1.) Pre facturacion tiempo entre alta medica (fecha de primera factura (fec_primera_fac) - fecha de salida (p_fec_sda) )
#### 2.) Ciclo de facturación (Tiempo entre primera y última factura (rework))
#### 3.) Transacciones/Correciones (Fricción financiera despues de la factura osea el que tanto se da el ciclo)
#### 4.) Requisiciones/Logística: vemos los trámites internos pendientes
#### 5.) Cierre administrativo: Tiempo desde ultima factura hasta la alta administrativa

In [6]:
# viendo que tantos datos faltantes hay
for i in range(len(nuevas_llaves)):
    print(f"Tabla: {nuevas_llaves[i]} valores faltantes: {dfs[nuevas_llaves[i]].isna().sum().sum()}")

Tabla: Hosder valores faltantes: 0
Tabla: Hosffa valores faltantes: 0
Tabla: Hosfol valores faltantes: 0
Tabla: Hoshab valores faltantes: 0
Tabla: Hospac valores faltantes: 3168
Tabla: Hosreq valores faltantes: 0
Tabla: Hostha valores faltantes: 0
Tabla: Hostransacciones valores faltantes: 160554


In [18]:
dfs['Hostransacciones']['internoexterno'].unique()

array([' ', 'E', None], dtype=object)

#### Nota acerca de la columna internoexterno:
#### interno externo: Paciente interno/externo


#### Importacion (poniendo nuestros jugetes (librerias) en nuestro tablero de batalla)

In [92]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

In [21]:
plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style("whitegrid")

In [38]:
# para facilidad de notacion
for i in nuevas_llaves:
    print(f"df_{i}=dfs['{i}']")

df_Hosder=dfs['Hosder']
df_Hosffa=dfs['Hosffa']
df_Hosfol=dfs['Hosfol']
df_Hoshab=dfs['Hoshab']
df_Hospac=dfs['Hospac']
df_Hosreq=dfs['Hosreq']
df_Hostha=dfs['Hostha']
df_Hostransacciones=dfs['Hostransacciones']


In [39]:
df_Hosder=dfs['Hosder']
df_Hosffa=dfs['Hosffa']
df_Hosfol=dfs['Hosfol']
df_Hoshab=dfs['Hoshab']
df_Hospac=dfs['Hospac']
df_Hosreq=dfs['Hosreq']
df_Hostha=dfs['Hostha']
df_Hostransacciones=dfs['Hostransacciones']


## ¿Qué tantos datos tenemos acerca del proceso?

In [59]:
# Dado que nuestro problema es temporal revisamos que tengamos los datos

# para Hospac
print("=== 1.1 COBERTURA TEMPORAL EN HOSPAC ===")
print(f"Registros HOSPAC: {len(df_Hospac):,}")
print(f"Con fecha llegada (p_fec_lld): {df_Hospac['p_fec_lld'].notna().sum():,}")
print(f"Con fecha alta médica (p_fec_sda): {df_Hospac['p_fec_sda'].notna().sum():,}")
print(f"Con fecha alta médica (p_fec_sda): {df_Hospac['p_fec_sda'].notna().sum():,}")

# para la tabla Hosfol
print("\n=== 1.2 COBERTURA DE LLAVES HOSFOL ===")
print(f"Folios con expediente (f_num_exp): {df_Hosfol['f_num_exp'].notna().sum():,}")

=== 1.1 COBERTURA TEMPORAL EN HOSPAC ===
Registros HOSPAC: 115,593
Con fecha llegada (p_fec_lld): 115,593
Con fecha alta médica (p_fec_sda): 115,593
Con fecha alta médica (p_fec_sda): 115,593

=== 1.2 COBERTURA DE LLAVES HOSFOL ===
Folios con expediente (f_num_exp): 113,325


In [60]:
print("\n=== 1.3 FACTURACIÓN: ¿1 factura o muchas? ===")
fac_por_folio = df_Hosffa.groupby(['folio','ext']).size()
print(f"Promedio facturas por folio: {fac_por_folio.mean():.2f}")
print(f"% de folios con >1 factura: {(fac_por_folio > 1).mean()*100:.1f}%")


=== 1.3 FACTURACIÓN: ¿1 factura o muchas? ===
Promedio facturas por folio: 1.17
% de folios con >1 factura: 13.3%


#### veamos que tanta friccion hay (pequeños fallos)

In [68]:
print("\n=== 1.4 TRANSACCIONES: ¿Hay fricción? ===")
if 'total' in df_Hostransacciones.columns:
    df_Hostransacciones['total_num'] = pd.to_numeric(df_Hostransacciones['total'], errors='coerce')
    neg = (df_Hostransacciones['total_num'] < 0).sum()
    print(f"Transacciones negativas: {neg:,} ({neg/len(df_Hostransacciones)*100:.1f}%)")


=== 1.4 TRANSACCIONES: ¿Hay fricción? ===
Transacciones negativas: 188,851 (28.8%)


In [64]:
print(f'Esto nos quiere decir que solo un {neg/len(df_Hostransacciones)*100:.2f}% son negatias en la base de datos')

Esto nos quiere decir que solo un 28.75% son negatias en la base de datos


### veamos Hosreq
#### Hosreq es la tabla de requesiciones
#### Veamos que tipo de valores tiene

In [123]:
print(set(df_Hosreq['st_req']))

{'01', '03', '05', '-1', '00'}


¿Que significa?
bueno estamos en el contexto de un hospital mexicano en el que se usan estos codigos numericos (en formato de string)
| Código | Probable significado                             |
| ------ | ------------------------------------------------ |
| `'00'` | Requisición **abierta / pendiente / solicitada** |
| `'01'` | Requisición **surtida / cerrada / completada**   |
| `'03'` | Requisición **parcialmente surtida**             |
| `'05'` | Requisición **cancelada**                        |
| `'-1'` | **Error / nulo / estado inválido** (legacy)      |


In [110]:
df_Hosreq.shape

(126810, 38)

In [111]:
# veamos la distribucion de estas categorias
conteo=df_Hosreq['st_req'].value_counts().reset_index()
conteo.columns=['st_req','count']
fig=px.pie(conteo,values='count',names='st_req')
print(conteo['count'])
fig.show()

0    81583
1    33048
2    12138
3       40
4        1
Name: count, dtype: int64


In [112]:
df_hosreq=df_Hosreq

In [114]:
print("\n=== 1.5 REQUISICIONES: ¿Hay pendientes? ===")
if 'st_req' in df_hosreq.columns:
    # Primero, veamos la distribución real
    print("Distribución de st_req:")
    print(df_hosreq['st_req'].value_counts().sort_index())

    # Definir qué códigos significan "pendiente" o "no resuelto"
    # Basado en convención típica de HIS mexicanos:
    estados_pendientes = {'00', '03'}  # abierta + parcial
    estados_cerrados = {'01'}          # surtida
    estados_cancelados = {'05'}        # cancelada
    estados_invalidos = {'-1'}         # nulo/error

    pendientes = df_hosreq['st_req'].isin(estados_pendientes).sum()
    cerradas = df_hosreq['st_req'].isin(estados_cerrados).sum()
    canceladas = df_hosreq['st_req'].isin(estados_cancelados).sum()
    invalidos = df_hosreq['st_req'].isin(estados_invalidos).sum()

    print(f"\nRequisiciones pendientes (00, 03):     {pendientes:,}")
    print(f"Requisiciones surtidas (01):           {cerradas:,}")
    print(f"Requisiciones canceladas (05):         {canceladas:,}")
    print(f"Requisiciones con estado inválido (-1): {invalidos:,}")

    # El flag que te interesa para cuello de botella:
    pct_pendientes = pendientes / len(df_hosreq) * 100
    print(f"\nPorcentaje de requisiciones NO cerradas: {pct_pendientes:.1f}%")


=== 1.5 REQUISICIONES: ¿Hay pendientes? ===
Distribución de st_req:
st_req
-1    12138
00       40
01    81583
03    33048
05        1
Name: count, dtype: int64

Requisiciones pendientes (00, 03):     33,088
Requisiciones surtidas (01):           81,583
Requisiciones canceladas (05):         1
Requisiciones con estado inválido (-1): 12,138

Porcentaje de requisiciones NO cerradas: 26.1%


In [118]:
print("\n=== 1.6 RESUMEN DE CALIDAD ===")
# Esto define tu poder de análisis
total = len(df_Hospac)
con_alta_med = df_Hospac['p_fec_sda'].notna().sum()
con_folio = df_Hosfol['f_num_exp'].notna().sum()
con_fac = df_Hosffa['folio'].notna().sum()
print(f"""
Episodios totales:              {total:,}
Con alta médica:                {con_alta_med:,} ({con_alta_med/total*100:.1f}%)
Con folio administrativo:       {con_folio:,}
Con al menos 1 factura:         {con_fac:,}
""")


=== 1.6 RESUMEN DE CALIDAD ===

Episodios totales:              115,593
Con alta médica:                115,593 (100.0%)
Con folio administrativo:       113,325
Con al menos 1 factura:         25,394



### Construccion de la tabla

### Nota:
#### p_fec_sda (alta médica) y alguna fecha de cierre/alta

In [167]:
df_Hospac['p_fec_sda']

0               NaT
1               NaT
2               NaT
3               NaT
4               NaT
            ...    
115588   2025-09-15
115589          NaT
115590   2025-11-05
115591          NaT
115592   2025-12-10
Name: p_fec_sda, Length: 115593, dtype: datetime64[ns]

In [134]:
df_pac=df_Hospac
df_pac['p_fec_lld'] = pd.to_datetime(df_pac['p_fec_lld'], errors='coerce')
df_pac['p_fec_sda'] = pd.to_datetime(df_pac['p_fec_sda'], errors='coerce')

In [159]:
pd.concat([df_pac['p_fec_lld'],df_pac['p_fec_sda']],axis=1).tail()

,p_fec_lld,p_fec_sda
115588,2025-09-15,2025-09-15
115589,2025-09-15,NaT
115590,2025-11-05,2025-11-05
115591,2025-11-21,NaT
115592,2025-12-10,2025-12-10


In [169]:
df_fol = df_Hosfol.copy()
df_fol['f_fec_ape'] = pd.to_datetime(df_fol['f_fec_ape'], errors='coerce')
df_fol['f_fec_cie'] = pd.to_datetime(df_fol['f_fec_cie'], errors='coerce')
pd.concat([df_fol['f_fec_ape'],df_fol['f_fec_cie']],axis=1)

,f_fec_ape,f_fec_cie
0,2024-12-16,2024-12-18
1,2024-12-16,2024-12-18
2,2024-12-19,2024-12-19
3,2024-12-19,2024-12-19
4,2024-12-19,2024-12-19
...,...,...
113320,2025-11-05,2025-11-05
113321,2025-11-21,NaT
113322,2025-11-21,NaT
113323,2025-11-24,2025-11-24


In [170]:
df_fac = df_Hosffa.copy()
df_fac['f_fac'] = pd.to_datetime(df_fac['f_fac'], errors='coerce')
df_fac['total_num'] = pd.to_numeric(df_fac['total'], errors='coerce')

df_trans = df_Hostransacciones.copy()
df_trans['fecha'] = pd.to_datetime(df_trans['fecha'], errors='coerce')
df_trans['total_num'] = pd.to_numeric(df_trans['total'], errors='coerce')


In [184]:
check=pd.concat([df_trans['fecha'],df_trans['total_num']],axis=1)
check[0:10]

,fecha,total_num
0,2024-12-18,175.0
1,2024-12-18,175.0
2,2024-12-19,175.0
3,2024-12-19,175.0
4,2024-12-20,175.0
5,2024-12-20,175.0
6,2024-12-30,1100.0
7,2024-12-30,1100.0
8,2025-01-02,300.0
9,2025-01-02,-300.0


In [198]:
fig=px.line(check[0:200],x='fecha',y='total_num')
fig.show()

In [199]:
df_req = df_hosreq.copy()
df_req['fec_sol'] = pd.to_datetime(df_req['fec_sol'], errors='coerce')

In [205]:
df_trans['cancelada'].unique()

array([' ', '1'], dtype=object)